# 01 · Exploring NASA OSDR data

This template shows how to load and summarise a **gene expression table** downloaded from the [NASA Open Science Data Repository](https://osdr.nasa.gov/).

**Before you start**
1. Search OSDR for a bacterial spaceflight study (e.g. keywords: *Bacillus subtilis*, *Pseudomonas*, *Salmonella*, *spaceflight*).
2. Download a differential-expression CSV from the study's files.
3. Save it as `data/raw/osdr_de.csv` (this folder is git-ignored; cite the OSDR study ID in the notes).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Update the path and column names after you inspect your file
PATH = 'data/raw/osdr_de.csv'
df = pd.read_csv(PATH)
print(df.shape)
df.head()

## Inspect columns
Column names differ between studies. Look for a gene ID, a log2 fold-change and an adjusted p-value.

In [ ]:
print(df.columns.tolist())
df.describe(include='all').T.head(20)

In [ ]:
# Rename to common names (EDIT the left-hand side to match your file)
df = df.rename(columns={
    'log2FoldChange': 'log2fc',
    'padj': 'padj',
})
df = df.dropna(subset=['log2fc', 'padj'])
len(df)

## Volcano plot
Genes with large fold-change and small adjusted p-value are the most interesting.

In [ ]:
import numpy as np

sig = (df['padj'] < 0.05) & (df['log2fc'].abs() > 1)
plt.figure(figsize=(7,5))
plt.scatter(df.loc[~sig,'log2fc'], -np.log10(df.loc[~sig,'padj']), s=8, alpha=0.5, label='not significant')
plt.scatter(df.loc[sig,'log2fc'], -np.log10(df.loc[sig,'padj']), s=10, alpha=0.8, label='significant')
plt.axvline(-1, ls='--', lw=0.8); plt.axvline(1, ls='--', lw=0.8)
plt.xlabel('log2 fold change (flight vs ground)')
plt.ylabel('-log10 adjusted p-value')
plt.title('Differential expression: spaceflight vs ground')
plt.legend()
plt.tight_layout()
plt.show()
print('Significant genes:', int(sig.sum()))

## Top genes
Examine the top up- and down-regulated genes and look up their function (UniProt, KEGG) to connect to biofilm, stress response or virulence.

In [ ]:
top_up = df[sig].sort_values('log2fc', ascending=False).head(10)
top_down = df[sig].sort_values('log2fc').head(10)
display(top_up); display(top_down)

## Interpretation (write in your own words)
- What biological processes are enriched?
- Is there evidence for changes in biofilm, stress or virulence genes?
- What are the limitations (sample size, hardware, strain, ground-control design)?